# Benchmark validity: GPU run

Trains and scores four new method groups on the 12 clinical references and 21 synthetic sets in the `wb-kit` dataset:

1. **sam**: SAM ViT-B and MedSAM ViT-B with box prompts (no training)
2. **pretrained**: ten architectures with ImageNet encoders, trained on FUSeg at 256 px
3. **kvasir**: the 14 trained methods retrained on Kvasir-SEG
4. **nnunet**: nnU-Net v2, 2D, trained on FUSeg

**Before running:** Settings, Accelerator: **GPU** (T4 x2 or P100). Settings, Internet: **On**. Add Input: the `wb-kit` dataset.
Then **Save Version, Save & Run All (Commit)**. It takes about 4 to 6 hours. When it finishes, open the version, go to Output, and download **wb_results.zip**.

In [ ]:
import glob, os, json
cands = glob.glob('/kaggle/input/**/manifest.json', recursive=True)
assert cands, 'Add the wb-kit dataset to this notebook (Add Input, Your Datasets)'
KIT = os.path.dirname(cands[0])
print('kit at', KIT)
os.environ['OUT'] = '/kaggle/working/out'
os.makedirs('/kaggle/working/out', exist_ok=True)

In [ ]:
!nvidia-smi
!pip install -q segmentation-models-pytorch==0.5.0 nnunetv2==2.8.1
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__)

In [ ]:
# 1. SAM and MedSAM with box prompts (about 20 minutes)
!cd "{KIT}" && python kit_run.py sam

In [ ]:
# 2. Ten ImageNet-pretrained architectures on FUSeg (about 40 minutes)
!cd "{KIT}" && python kit_run.py pretrained

In [ ]:
# 3. The 14 trained methods retrained on Kvasir-SEG (about 60 minutes)
!cd "{KIT}" && python kit_run.py kvasir

In [ ]:
# 4. nnU-Net 2D on FUSeg (about 2 to 3 hours)
!cd "{KIT}" && python kit_run.py nnunet

In [ ]:
# 5. Pack the results
!cd "{KIT}" && python kit_run.py pack
!ls -la /kaggle/working
print(open('/kaggle/working/out/log.txt').read()[-6000:])